# 03_extraccion_ecg_pdf

**Objetivo:** reproducir el proceso de extracción estructurada de datos desde reportes ECG en formato PDF.

Este notebook transforma reportes ECG almacenados como PDFs en un dataset tabular estructurado, incorporando parámetros electrocardiográficos, metadatos básicos del examen, controles de validez, completitud y claves de matching para integración posterior con la cohorte clínica procesada.

**Entradas esperadas:**

- Carpeta local con PDFs ECG.
- Estructura opcional de carpetas por año, mes y día.

**Salidas generadas:**

- `ecg_dataset.csv`
- `ecg_dataset.xlsx`
- `ecg_resumen.txt`

**Uso dentro del TFM:**

Este notebook antecede a la integración con la cohorte clínica. Su salida principal, `ecg_dataset.xlsx`, será utilizada por el notebook posterior de integración multimodal.

## 1. Librerías y configuración inicial

El proceso utiliza `pypdfium2` para extraer texto desde PDFs nativos. No se realiza OCR. Por tanto, el notebook asume que los reportes ECG contienen texto embebido extraíble.

In [ ]:
from pathlib import Path
from datetime import datetime
import glob
import os
import re
import sys
import traceback
import unicodedata

import numpy as np
import pandas as pd
import pypdfium2 as pdfium

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

## 2. Parámetros de ejecución

Ajustar `PDF_ROOT` a la carpeta donde se encuentran los reportes ECG en PDF. El notebook busca archivos PDF de forma recursiva.

La variable `OUT_DIR` define la carpeta donde se escribirán las salidas tabulares y el resumen técnico.

In [ ]:
# Carpeta raíz que contiene los PDFs ECG.
# Ejemplo Windows: Path(r"C:/Users/viggo/ZCodeProject/ELECTROCARDIOGRAMA")
PDF_ROOT = Path("ELECTROCARDIOGRAMA")

# Carpeta de salida.
OUT_DIR = Path("outputs_ecg")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Si se ejecuta en una carpeta de pruebas sin subcarpeta ELECTROCARDIOGRAMA,
# se permite usar el directorio actual cuando contiene PDFs.
if not PDF_ROOT.exists():
    local_pdfs = list(Path(".").glob("*.pdf"))
    if local_pdfs:
        PDF_ROOT = Path(".")

PDF_ROOT, OUT_DIR

## 3. Diccionarios y expresiones regulares base

`MESES_ES` permite recuperar el mes desde rutas con carpetas en español. `NUM` captura números enteros o decimales, tolerando coma decimal y espacios internos generados por segmentación del texto del PDF.

In [ ]:
MESES_ES = {
    "ENERO": "01", "FEBRERO": "02", "MARZO": "03", "ABRIL": "04",
    "MAYO": "05", "JUNIO": "06", "JULIO": "07", "AGOSTO": "08",
    "SEPTIEMBRE": "09", "OCTUBRE": "10", "NOVIEMBRE": "11", "DICIEMBRE": "12",
}

NUM = r"(-?\d[\d ,.]*\d|-?\d)"

CORE_ECG_COLS = ["ECG_HR", "ECG_PR", "ECG_QRS", "ECG_QTC", "ECG_AXIS"]

## 4. Funciones auxiliares de normalización

Estas funciones convierten números a formato estándar, normalizan nombres de archivo para matching y transforman fechas de examen desde formato `DD-MM-YYYY` hacia `YYYY-MM-DD`.

In [ ]:
def to_num(val):
    """Convierte valores numéricos extraídos desde texto PDF a int/float."""
    if val is None:
        return None
    if isinstance(val, (int, float, np.integer, np.floating)):
        return val.item() if hasattr(val, "item") else val
    s = str(val).strip()
    if s == "" or s.upper() == "NULL":
        return None
    s = s.replace(" ", "")
    s = s.replace(",", ".")
    try:
        f = float(s)
        return int(f) if f.is_integer() else f
    except Exception:
        return None


def normalize_name(name):
    """Normaliza nombre de archivo para matching: mayúsculas, sin acentos ni sufijos duplicados."""
    if not name:
        return None
    s = os.path.basename(str(name))
    s = re.sub(r"\.pdf$", "", s, flags=re.I)
    s = re.sub(r"[\s_\-]+(\d+)\s*$", "", s)
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = re.sub(r"\s+", " ", s).strip().upper()
    return s or None


def fecha_to_iso(fecha_examen):
    """Convierte 'DD-MM-YYYY HH:MM:SS' a 'YYYY-MM-DD'."""
    if not fecha_examen or not isinstance(fecha_examen, str):
        return None
    m = re.search(r"(\d{2})-(\d{2})-(\d{4})", fecha_examen)
    if not m:
        return None
    dd, mm, yyyy = m.group(1), m.group(2), m.group(3)
    try:
        datetime(int(yyyy), int(mm), int(dd))
        return f"{yyyy}-{mm}-{dd}"
    except ValueError:
        return None


def search(pat, text, flags=re.IGNORECASE):
    m = re.search(pat, text, flags)
    return m.group(1).strip() if m else None


def search2(pat, text, flags=re.IGNORECASE):
    m = re.search(pat, text, flags)
    if not m:
        return None, None
    return m.group(1).strip(), m.group(2).strip()

## 5. Extracción de texto desde PDF

La función `extract_text` consolida el texto de todas las páginas. Se genera una versión cruda y una versión normalizada con saltos de línea reemplazados por espacios, lo que permite capturar parámetros fragmentados por el renderizado del PDF.

In [ ]:
def extract_text(pdf_path):
    """Extrae texto crudo y texto normalizado desde un PDF."""
    doc = pdfium.PdfDocument(str(pdf_path))
    parts = []
    for i in range(len(doc)):
        page = doc[i]
        tp = page.get_textpage()
        txt = tp.get_text_range()
        parts.append(txt)
        tp.close()
        page.close()
    doc.close()

    raw = "
".join(parts)
    norm = re.sub(r"[
]+", " ", raw)
    norm = re.sub(r"\s+", " ", norm).strip()
    return raw, norm

## 6. Limpieza de texto clínico residual

Los reportes ECG pueden contener trazados, derivaciones, cabeceras y fragmentos numéricos que no corresponden a prosa clínica. Estas funciones permiten aislar texto potencialmente informativo para `ECG_ANALYSIS` y `ECG_DIAGNOSIS`.

In [ ]:
def is_noise(line):
    """Identifica líneas sin valor clínico textual para analysis/diagnosis."""
    s = str(line).strip()
    if not s:
        return True
    if re.fullmatch(r"[\d\s,./]+", s) and re.search(r"\d", s) and len(re.findall(r"\d", s)) > len(s) * 0.5:
        return True
    if re.fullmatch(r"((I{1,3}|aVR|aVL|aVF|V[1-6])\s*)+", s):
        return True
    if re.match(r"^(HR|PR|QRS|QT|QTc|RV\d|SV\d|QRS Axis|Name|Pati|ECG No|Age|Sex)", s, re.I):
        return True
    if "ECG Report" in s or "Rpt.Date" in s or "Exam/Dr" in s:
        return True
    if re.match(r"\d{2}-\d{2}-\d{4}", s):
        return True
    if "10mm/mV" in s or "25mm/s" in s:
        return True
    if "ECG Parameters" in s or "ECG Analysis" in s or "ECG Diagnosis" in s:
        return True
    if re.fullmatch(r"(bpm|bmp|ms|mV|Deg\.?)", s, re.I):
        return True
    if re.fullmatch(r"=?\s*-?\d[\d ,.]*\s*(ms|mV|Deg\.?|bpm|bmp)?", s, re.I):
        return True
    return False


def looks_like_clinical_prose(line):
    """Heurística para conservar solamente texto tipo prosa clínica."""
    s = str(line).strip()
    if not s or is_noise(s):
        return False
    letters = re.findall(r"[A-Za-zÁÉÍÓÚáéíóúÑñ]{2,}", s)
    if len(letters) < 2:
        return False
    long_words = [w for w in letters if len(w) >= 4]
    noise_words = {"Undefined", "HospitalECG"}
    long_words = [w for w in long_words if w not in noise_words]
    return len(long_words) >= 1


def extract_analysis_diagnosis(text):
    """Extrae prosa clínica residual desde el bloque analysis/diagnosis cuando existe."""
    lines = [l.rstrip() for l in str(text).splitlines()]
    textual = []
    for line in lines:
        s = line.strip()
        if not s:
            continue
        if not looks_like_clinical_prose(s):
            continue
        textual.append(s)
    blob = " | ".join(textual).strip()
    if not blob:
        return None, None
    return None, blob

## 7. Parsing estructurado de un ECG PDF

La función `parse_ecg` extrae metadatos del archivo, fecha de examen, edad, sexo, parámetros ECG y claves de matching. Los campos `pdf_valido`, `parametros_extraidos` y `observaciones_extraccion` permiten auditar la calidad del proceso.

In [ ]:
def empty_record():
    return {
        "archivo_origen": None, "ruta_relativa": None, "año": None, "mes": None,
        "fecha_examen": None, "paciente_id": None, "sexo": None, "edad": None,
        "ECG_HR": None, "ECG_PR": None, "ECG_QRS": None, "ECG_QTC": None,
        "ECG_AXIS": None, "QT": None, "QRS_AXIS": None,
        "RV5": None, "SV1": None, "RV1": None, "SV5": None,
        "ECG_ANALYSIS": None, "ECG_DIAGNOSIS": None,
        "pdf_valido": 0, "parametros_extraidos": 0,
        "observaciones_extraccion": "",
        "nombre_paciente": None, "nombre_paciente_norm": None,
        "fecha": None, "clave_matching": None,
        "num_ecg_mismo_paciente_fecha": 0, "duplicado_mismo_dia": 0,
    }


def parse_ecg(pdf_path, root=PDF_ROOT):
    rec = empty_record()
    obs = []
    pdf_path = Path(pdf_path)
    root = Path(root)

    try:
        try:
            rel = pdf_path.relative_to(root)
        except ValueError:
            rel = Path(os.path.relpath(pdf_path, root))

        rec["ruta_relativa"] = str(rel).replace("\", "/")
        rec["archivo_origen"] = pdf_path.name

        parts = [p for p in rec["ruta_relativa"].split("/") if p]
        year_idx = None
        for idx, p in enumerate(parts):
            if re.fullmatch(r"\d{4}", p) and int(p) >= 1900:
                rec["año"] = p
                year_idx = idx
                break
        if year_idx is not None:
            for p in parts[year_idx + 1:]:
                up = p.upper()
                if up in MESES_ES:
                    rec["mes"] = MESES_ES[up]
                    break
                if re.fullmatch(r"\d{2}", p) and 1 <= int(p) <= 12:
                    rec["mes"] = p
                    break

        text, norm = extract_text(pdf_path)
        if not text or not text.strip():
            rec["observaciones_extraccion"] = "PDF sin texto extraible"
            return rec
        rec["pdf_valido"] = 1

        m = re.search(r"Pati\.?ID\s*(\S+)", norm, re.I)
        if m:
            rec["paciente_id"] = m.group(1).strip()

        m = re.search(r"Name\s+(.*?)\s*Sex\s*([MFmf])\s*Age\s*(\d+)\s*Y", norm, re.I | re.S)
        if m:
            rec["sexo"] = m.group(2).upper()
            rec["edad"] = to_num(m.group(3))
        else:
            ms = re.search(r"Sex\s*([MFmf])", norm, re.I)
            me = re.search(r"Age\s*(\d+)\s*Y", norm, re.I)
            if ms:
                rec["sexo"] = ms.group(1).upper()
            if me:
                rec["edad"] = to_num(me.group(1))
            if not ms or not me:
                obs.append("bloque Name/Sex/Age no coincide con patron completo")

        mf = re.search(r"(\d{2}-\d{2}-\d{4}\s+\d{1,2}:\d{2}(?::\d{2})?)", norm)
        if mf:
            rec["fecha_examen"] = mf.group(1).strip()

        rec["nombre_paciente"] = pdf_path.stem
        rec["nombre_paciente_norm"] = normalize_name(pdf_path.name)
        rec["fecha"] = fecha_to_iso(rec["fecha_examen"])
        if rec["nombre_paciente_norm"] and rec["fecha"]:
            rec["clave_matching"] = f"{rec['nombre_paciente_norm']} | {rec['fecha']}"

        rec["ECG_HR"] = to_num(search(r"HR\s*=?\s*" + NUM + r"\s*(?:bpm|bmp|b/min)?", norm))
        rec["ECG_PR"] = to_num(search(r"PR\s*=?\s*" + NUM + r"\s*ms", norm))
        rec["ECG_QRS"] = to_num(search(r"QRS\s*=?\s*" + NUM + r"\s*ms", norm))

        axis = search(r"QRS\s*Axis\s*=?\s*" + NUM, norm)
        rec["ECG_AXIS"] = to_num(axis)
        rec["QRS_AXIS"] = rec["ECG_AXIS"]

        qt, qtc = search2(r"QT\s*/\s*QTc\s*=?\s*" + NUM + r"\s*/\s*" + NUM, norm)
        if qt is not None:
            rec["QT"] = to_num(qt)
            rec["ECG_QTC"] = to_num(qtc)
        else:
            rec["ECG_QTC"] = to_num(search(r"QTc\s*=?\s*" + NUM, norm))

        rv5, sv1 = search2(r"RV5\s*/\s*SV1\s*=?\s*" + NUM + r"\s*/\s*" + NUM, norm)
        if rv5 is not None:
            rec["RV5"] = to_num(rv5)
            rec["SV1"] = to_num(sv1)

        rv1, sv5 = search2(r"RV1\s*/\s*SV5\s*=?\s*" + NUM + r"\s*/\s*" + NUM, norm)
        if rv1 is not None:
            rec["RV1"] = to_num(rv1)
            rec["SV5"] = to_num(sv5)

        analysis, diagnosis = extract_analysis_diagnosis(text)
        rec["ECG_ANALYSIS"] = analysis
        rec["ECG_DIAGNOSIS"] = diagnosis

        rec["parametros_extraidos"] = sum(1 for c in CORE_ECG_COLS if rec[c] is not None)
        missing = [k for k, c in zip(["HR", "PR", "QRS", "QTc", "Axis"], CORE_ECG_COLS) if rec[c] is None]
        if missing:
            obs.append("parametros faltantes: " + ",".join(missing))

        rec["observaciones_extraccion"] = "; ".join(o for o in obs if o)

    except Exception as e:
        rec["pdf_valido"] = 0
        rec["observaciones_extraccion"] = "ERROR: " + repr(e)[:300]

    return rec

## 8. Detección de PDFs y procesamiento batch

Se procesan todos los archivos `.pdf` encontrados bajo `PDF_ROOT`. Si no se detectan archivos, revisar la ruta configurada en la sección 2.

In [ ]:
pdfs = sorted(PDF_ROOT.rglob("*.pdf"))
print(f"PDF_ROOT: {PDF_ROOT.resolve()}")
print(f"PDFs detectados: {len(pdfs)}")

pdfs[:5]

In [ ]:
rows = []

for i, pdf_path in enumerate(pdfs, 1):
    rec = parse_ecg(pdf_path, root=PDF_ROOT)
    rows.append(rec)
    if i % 25 == 0 or i == len(pdfs):
        print(f"Procesados {i}/{len(pdfs)} PDFs")

cols = [
    "archivo_origen", "ruta_relativa", "año", "mes", "fecha_examen",
    "paciente_id", "sexo", "edad", "ECG_HR", "ECG_PR", "ECG_QRS", "ECG_QTC",
    "ECG_AXIS", "QT", "QRS_AXIS", "RV5", "SV1", "RV1", "SV5",
    "ECG_ANALYSIS", "ECG_DIAGNOSIS", "pdf_valido", "parametros_extraidos",
    "observaciones_extraccion", "nombre_paciente", "nombre_paciente_norm", "fecha",
    "clave_matching", "num_ecg_mismo_paciente_fecha", "duplicado_mismo_dia"
]

df_ecg = pd.DataFrame(rows, columns=cols)
df_ecg.head()

## 9. Control de duplicados por paciente y fecha

La clave de matching operacional se construye como `nombre_paciente_norm + fecha`. Esta clave permite identificar múltiples ECG asociados al mismo paciente en el mismo día.

In [ ]:
if not df_ecg.empty:
    counts = df_ecg.groupby("clave_matching")["clave_matching"].transform("count")
    df_ecg["num_ecg_mismo_paciente_fecha"] = counts.fillna(0).astype(int)
    df_ecg["duplicado_mismo_dia"] = (
        (df_ecg["num_ecg_mismo_paciente_fecha"] > 1) & df_ecg["clave_matching"].notna()
    ).astype(int)

    display(df_ecg[["archivo_origen", "fecha", "clave_matching", "num_ecg_mismo_paciente_fecha", "duplicado_mismo_dia"]].head(10))
else:
    print("No hay registros ECG procesados.")

## 10. Validación de completitud de parámetros ECG

Se evalúa la completitud de los cinco parámetros core utilizados en el estudio: frecuencia cardiaca, PR, QRS, QTc y eje QRS.

In [ ]:
if not df_ecg.empty:
    df_ecg["ECG_CORE_COMPLETO"] = df_ecg[CORE_ECG_COLS].notna().all(axis=1).astype(int)

    resumen_core = pd.DataFrame({
        "variable": CORE_ECG_COLS,
        "registros_no_nulos": [int(df_ecg[c].notna().sum()) for c in CORE_ECG_COLS],
        "porcentaje_completitud": [round(df_ecg[c].notna().mean() * 100, 2) for c in CORE_ECG_COLS],
    })

    display(resumen_core)
    print("ECG con parámetros core completos:", int(df_ecg["ECG_CORE_COMPLETO"].sum()))
else:
    print("No hay registros ECG procesados.")

## 11. Exportación de resultados

Se generan las salidas tabulares y el resumen técnico del proceso.

In [ ]:
def build_resumen(df):
    total = len(df)
    lines = []
    lines.append("RESUMEN EXTRACCION ECG")
    lines.append("=" * 50)
    lines.append(f"Fecha generacion : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    lines.append(f"PDF_ROOT         : {PDF_ROOT.resolve()}")
    lines.append(f"OUT_DIR          : {OUT_DIR.resolve()}")
    lines.append(f"Total PDFs procesados         : {total}")

    if total == 0:
        lines.append("Sin registros procesados.")
        return "
".join(lines)

    validos = int((df["pdf_valido"] == 1).sum())
    errores = int((df["pdf_valido"] == 0).sum())
    completos = int(df[CORE_ECG_COLS].notna().all(axis=1).sum())

    lines.append(f"Total ECG validos             : {validos}")
    lines.append(f"Total ECG con errores          : {errores}")
    lines.append(f"ECG con 5 parametros core OK   : {completos}")
    lines.append("")
    lines.append("COMPLETITUD POR VARIABLE (%)")
    lines.append("-" * 50)

    var_cols = [
        "paciente_id", "sexo", "edad", "ECG_HR", "ECG_PR", "ECG_QRS", "ECG_QTC",
        "ECG_AXIS", "QT", "RV5", "SV1", "RV1", "SV5", "ECG_ANALYSIS", "ECG_DIAGNOSIS"
    ]
    for c in var_cols:
        nn = int(df[c].notna().sum()) if c in df.columns else 0
        pct = round(nn / total * 100, 1) if total else 0
        lines.append(f"{c:18s}: {pct:5.1f}%  ({nn}/{total})")

    lines.append("")
    lines.append("DISTRIBUCION POR ANIO")
    lines.append("-" * 50)
    for k, v in df["año"].value_counts(dropna=False).sort_index().items():
        lines.append(f"  {k}: {v}")

    lines.append("")
    lines.append("DISTRIBUCION POR MES")
    lines.append("-" * 50)
    for k, v in df["mes"].value_counts(dropna=False).sort_index().items():
        lines.append(f"  {k}: {v}")

    lines.append("")
    lines.append("DISTRIBUCION DE parametros_extraidos")
    lines.append("-" * 50)
    for n in range(6):
        cnt = int((df["parametros_extraidos"] == n).sum())
        lines.append(f"  {n} parametros: {cnt}")

    lines.append("")
    lines.append("MATCHING PACIENTE")
    lines.append("-" * 50)
    lines.append("  Estrategia             : nombre_paciente_norm + fecha")
    lines.append(f"  Pacientes unicos       : {df['nombre_paciente_norm'].nunique(dropna=True)}")
    lines.append(f"  Claves matching unicas : {df['clave_matching'].nunique(dropna=True)}")
    lines.append(f"  Registros duplicados   : {int(df['duplicado_mismo_dia'].sum())}")

    lines.append("")
    lines.append("OBSERVACIONES DE EXTRACCION")
    lines.append("-" * 50)
    nz = df[df["observaciones_extraccion"].astype(str).str.len() > 0]
    if len(nz) == 0:
        lines.append("  (sin observaciones)")
    else:
        for _, r in nz.iterrows():
            lines.append(f"  - {r['archivo_origen']}: {r['observaciones_extraccion']}")

    return "
".join(lines)


csv_path = OUT_DIR / "ecg_dataset.csv"
xlsx_path = OUT_DIR / "ecg_dataset.xlsx"
txt_path = OUT_DIR / "ecg_resumen.txt"

if not df_ecg.empty:
    df_ecg.to_csv(csv_path, index=False, encoding="utf-8-sig")
    df_ecg.to_excel(xlsx_path, index=False)

resumen = build_resumen(df_ecg)
txt_path.write_text(resumen, encoding="utf-8")

print(resumen)
print("
Archivos generados:")
print(csv_path)
print(xlsx_path)
print(txt_path)

## 12. Consideraciones de privacidad y reproducibilidad

- Los PDFs ECG originales pueden contener identificadores personales y no deben publicarse en repositorios abiertos.
- `ecg_dataset.xlsx` puede conservar información reidentificable si contiene nombres, fechas o rutas derivadas de archivos reales.
- Para publicación en GitHub se recomienda incluir solo notebooks, scripts, documentación, diccionarios y datos sintéticos.
- El campo `paciente_id` del PDF no debe asumirse como identificador clínico real. La clave operacional para cruce posterior es `nombre_paciente_norm + fecha`.
- Este proceso no constituye una herramienta clínica validada. Su uso es académico y experimental dentro del TFM.